In [1]:
import os
from dotenv import load_dotenv

load_dotenv() # This looks for the .env file and loads it
print("Kitchen is ready! 🍳")

Kitchen is ready! 🍳


In [4]:
# Cell 2: Reading the raw text
file_path = "../data/outputs/lecture_transcript.txt" # Adjust this if your notebook is in a different relative location

# We use 'with open' to safely open the file
with open(file_path, "r", encoding="utf-8") as file:
    raw_text = file.read()

print("--- RAW TEXT ---")
print(raw_text)

--- RAW TEXT ---
# 1-Hour Lecture Transcript — Computer Networks

### Topic: Internet Protocols and Web Communication

**Style:** Highly academic, concept-heavy classroom transcript
**Focus:** Definitions + conceptual explanations + examples + key points + realistic classroom sounds
**Duration:** ~60 minutes

---

## [00:00 – 05:00] Introduction to Computer Networks

**[Background: ceiling fan humming. Students talking quietly. Chairs moving.]**

**Professor:** Good morning, everyone. Settle down, please.

**Students:** Good morning, sir.

**Professor:** Today we're going to study something extremely important in Computer Networks — **network protocols and Internet communication**.

Before we begin with individual protocols, let's establish some definitions.

**[Marker clicks.]**

### Definition: Computer Network

A **computer network** is a collection of interconnected computing devices that communicate with each other and exchange data using defined communication protocols.

Now noti

In [5]:
import re

def clean_transcript(text):
    # 1. Replace any multiple whitespace (spaces, tabs, newlines) with a single space
    cleaned = re.sub(r'\s+', ' ', text)
    
    # 2. Strip leading and trailing spaces from the very beginning/end
    cleaned = cleaned.strip()
    
    return cleaned

clean_text = clean_transcript(raw_text)

print("--- CLEANED TEXT ---")
print(clean_text)

--- CLEANED TEXT ---
# 1-Hour Lecture Transcript — Computer Networks ### Topic: Internet Protocols and Web Communication **Style:** Highly academic, concept-heavy classroom transcript **Focus:** Definitions + conceptual explanations + examples + key points + realistic classroom sounds **Duration:** ~60 minutes --- ## [00:00 – 05:00] Introduction to Computer Networks **[Background: ceiling fan humming. Students talking quietly. Chairs moving.]** **Professor:** Good morning, everyone. Settle down, please. **Students:** Good morning, sir. **Professor:** Today we're going to study something extremely important in Computer Networks — **network protocols and Internet communication**. Before we begin with individual protocols, let's establish some definitions. **[Marker clicks.]** ### Definition: Computer Network A **computer network** is a collection of interconnected computing devices that communicate with each other and exchange data using defined communication protocols. Now notice three 

In [4]:
import whisper
import warnings

# This suppresses some annoying warnings about audio processing
warnings.filterwarnings("ignore")

# 1. Load the stenographer's brain. 
# We use "base" because it's a good balance of speed and accuracy for laptops.
print("Loading Whisper model (this takes a moment the first time)...")
model = whisper.load_model("base")
print("Model loaded!")

# 2. Transcribe the audio
audio_path = "../data/raw_audio/test_audio.mp3"
print(f"Listening to {audio_path}...")
# This step might take a minute depending on your computer's speed!
result = model.transcribe(audio_path)

# 3. Extract the text
transcribed_text = result["text"]
print("\n--- TRANSCRIBED TEXT ---")
print(transcribed_text)

Loading Whisper model (this takes a moment the first time)...
Model loaded!
Listening to ../data/raw_audio/test_audio.mp3...

--- TRANSCRIBED TEXT ---
 The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun.


In [6]:
# Cell 4: Intelligently Chunking the Text
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Configure our splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,       # Max number of characters per chunk
    chunk_overlap=500,     # How many characters to overlap between chunks
    length_function=len,   # How we measure length (characters)
    separators=["\n\n", "\n", ".", " ", ""] # The order of preference for splitting
)

# 2. Perform the split
# (Assuming 'transcribed_text' is the variable from your Whisper cell)
chunks = text_splitter.split_text(clean_text)

# 3. Print the results
print(f"Total chunks created: {len(chunks)}\n")

# Let's look at the first two chunks to see the overlap in action
for i, chunk in enumerate(chunks[:2]): # Just checking the first 2
    print(f"--- CHUNK {i+1} (Length: {len(chunk)}) ---")
    print(chunk)
    print("\n")

Total chunks created: 5

--- CHUNK 1 (Length: 3859) ---
# 1-Hour Lecture Transcript — Computer Networks ### Topic: Internet Protocols and Web Communication **Style:** Highly academic, concept-heavy classroom transcript **Focus:** Definitions + conceptual explanations + examples + key points + realistic classroom sounds **Duration:** ~60 minutes --- ## [00:00 – 05:00] Introduction to Computer Networks **[Background: ceiling fan humming. Students talking quietly. Chairs moving.]** **Professor:** Good morning, everyone. Settle down, please. **Students:** Good morning, sir. **Professor:** Today we're going to study something extremely important in Computer Networks — **network protocols and Internet communication**. Before we begin with individual protocols, let's establish some definitions. **[Marker clicks.]** ### Definition: Computer Network A **computer network** is a collection of interconnected computing devices that communicate with each other and exchange data using defined communi

In [8]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv() 
client = genai.Client()

first_chunk = chunks[0]

prompt = f"""
# Role
You are an expert University Teaching Assistant specializing in **Academic Distillation**. Your sole objective is to extract the intellectual content from a lecture transcript and convert it into **linear, high-fidelity study notes**.

# Critical Filtering Rules
1. **STRICT ACADEMIC FILTER:** You must aggressively discard ALL non-academic content. 
   - ❌ IGNORE: Greetings, roll call, tech checks ("Can you hear me?"), administrative announcements, personal anecdotes unrelated to the theory, jokes, and filler words.
   - ✅ KEEP: Definitions, theories, arguments, evidence, case studies, formulas, dates, names of scholars, and logical progressions.
2. **NO HALLUCINATION:** Do not add external knowledge or context not explicitly present in the text. If the professor is vague, note the vagueness; do not fill in the blanks yourself.
3. **LINEAR FLOW:** Structure the notes chronologically as they appear in the lecture. Do not reorganize into thematic clusters unless the professor explicitly does so.

# Output Format
Adhere strictly to this structure:

### 🎓 Lecture Thesis
(A single, dense paragraph stating the central argument or learning objective of the session. No fluff.)

### 📖 Linear Academic Notes
(Follow the lecture's progression. Use bolding for concepts and indentation for supporting details.)

**[Topic/Concept Name]**
- **Core Definition/Argument:** [Precise explanation using the professor's terminology]
- **Supporting Evidence/Example:** [Specific data, case studies, or analogies cited]
- **Logical Connection:** [How this point advances the previous argument or sets up the next one]

*(Repeat this block for every distinct academic segment in the transcript)*

### 🔑 Essential Terminology
(Only terms defined or heavily used in the lecture)
- **[Term]:** [Definition] — *Contextual relevance:* [Why this term was important to the specific argument made]

### ⚠️ Ambiguities & Unresolved Points
(List any questions the professor posed but did not answer, contradictions in the lecture, or complex points that were rushed. This helps the student know what to research independently.)

--- TRANSCRIPT CHUNK ---
{first_chunk}
"""

# Our list of models in order of preference
fallback_models = ['gemini-3.5-flash', 'gemini-3.6-flash', 'gemini-3.7-flash' , 'gemini-3.8-flash']

print("Starting AI generation...\n")

for model_name in fallback_models:
    print(f"Attempting to use {model_name}...")
    
    try:
        # We TRY to call the API
        response = client.models.generate_content(
            model=model_name,
            contents=prompt
        )
        
        print(f"✅ Success with {model_name}!\n")
        print("--- AI GENERATED NOTES ---")
        print(response.text)
        
        # We got our answer, so we 'break' out of the loop! We don't need to try the others.
        break 
        
    except Exception as e:
        # If it fails, Python jumps here INSTEAD of crashing with a red error!
        print(f"❌ {model_name} failed. Error: {e}")
        print("Switching to the next model...\n")
        
else:
    # Fun fact: In Python, you can put an 'else' on a 'for' loop. 
    # It only runs if the loop finished WITHOUT ever hitting 'break'.
    print("🚨 All models failed. Check your .env API key!")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Starting AI generation...

Attempting to use gemini-3.5-flash...
✅ Success with gemini-3.5-flash!

--- AI GENERATED NOTES ---
### 🎓 Lecture Thesis
The structural and functional integrity of computer networks relies on the integration of interconnected nodes communicating via standardized protocols—governed by syntax, semantics, and timing. Data transmission across these networks requires segmenting raw data into formatted packets (comprising a header and a payload) and processing them through a hierarchical protocol stack via sequential encapsulation and decapsulation.

---

### 📖 Linear Academic Notes

**Computer Network**
- **Core Definition/Argument:** A computer network is a collection of interconnected computing devices that exchange data and communicate with each other. For meaningful communication to occur, devices must be both physically/logically connected and share a common communication protocol.
- **Supporting Evidence/Example:** The definition rests on three core pillars: 

In [9]:
from pydantic import BaseModel
from google.genai import types
# 1. New strict blueprint for a single Key Term
class KeyTerm(BaseModel):
    term: str
    definition: str
# 2. Strict blueprint for a Concept
class ConceptDetail(BaseModel):
    concept_name: str
    detailed_explanation: str 
    examples: list[str]       
# 3. Our main blueprint now only uses strict lists!
class StudyNotes(BaseModel):
    lecture_title: str
    comprehensive_summary: str
    key_terms: list[KeyTerm]           # <-- FIXED! No more dictionary!
    core_concepts: list[ConceptDetail]

# 3. We upgrade our Prompt to demand detail
json_prompt = f"""
You are an expert university professor creating an exhaustive, detailed study guide.
Read the following transcript chunk and extract the information in depth.

RULES:
- Be EXHAUSTIVE. Do not skip key details, nuances, or examples.
- Write explanations as if you are teaching a beginner.
- Do NOT invent or hallucinate any information. Only use the facts provided.

--- TRANSCRIPT CHUNK ---
{chunks[0]}
"""

print("Asking Gemini to generate EXHAUSTIVE notes...")

try:
    response = client.models.generate_content(
        model='gemini-3.5-flash', 
        contents=json_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=StudyNotes,
        ),
    )
except Exception:
    print("Pro failed, falling back to Flash...")
    response = client.models.generate_content(
        model='gemini-3.6-flash', 
        contents=json_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=StudyNotes,
        ),
    )

my_detailed_notes = response.parsed

print("\n✅ Success! Let's look at one of the detailed concepts:\n")

# Let's print just the first core concept to see how much better it is
if my_detailed_notes.core_concepts:
    first_concept = my_detailed_notes.core_concepts[0]
    print(f"📌 {first_concept.concept_name}")
    print(f"Explanation: {first_concept.detailed_explanation}")
    print(f"Examples: {first_concept.examples}")

Asking Gemini to generate EXHAUSTIVE notes...

✅ Success! Let's look at one of the detailed concepts:

📌 The Three Pillars of Protocol Design
Explanation: For any network protocol to function successfully, it must define three core aspects of communication: Syntax, Semantics, and Timing. Syntax refers to the structure or format of the data, dictating what the message looks like. Semantics refers to the meaning of each section of bits or fields, explaining how they should be interpreted. Timing defines when data should be sent and how fast it can flow, preventing a fast sender from overwhelming a slow receiver.
Examples: ['Syntax: Specifying that the first 8 bits of a packet represent the address.', 'Semantics: Defining that an address ending in all 1s means it is a broadcast message.', 'Timing: Establishing a transmission window or speed coordination between sender and receiver.']


In [12]:
import time
import os

all_chunk_notes = []
print(f"Starting MAP phase: Processing {len(chunks)} chunks...\n")

# --- THE MAP PHASE ---
for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i+1} / {len(chunks)}...")
    
    prompt = f"""
    You are an expert university professor creating an exhaustive, detailed study guide.
    Read the following transcript chunk and extract the information in depth.
    RULES: Do NOT invent information. Only use the facts provided.
    
    --- TRANSCRIPT CHUNK ---
    {chunk}
    """
    
    try:
        # Using YOUR confirmed working model!
        response = client.models.generate_content(
            model='gemini-3.5-flash-lite', 
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=StudyNotes,
            ),
        )
        all_chunk_notes.append(response.parsed)
        time.sleep(4) # Respect the free tier rate limit!
        
    except Exception as e:
        print(f"Failed on chunk {i+1}. Error: {e}")

Starting MAP phase: Processing 5 chunks...

Processing chunk 1 / 5...
Processing chunk 2 / 5...
Processing chunk 3 / 5...
Processing chunk 4 / 5...
Processing chunk 5 / 5...


In [13]:
# Cell 8: Phase 7 - Reduce and Export to Markdown
import os

print("Combining notes and removing duplicates...")

# --- THE REDUCE PHASE ---
master_summary = ""
unique_terms_dict = {}
unique_concepts_dict = {}

for note in all_chunk_notes:
    # Combine summaries
    master_summary += note.comprehensive_summary + "\n\n"
    
    # Deduplicate Key Terms
    for kt in note.key_terms:
        unique_terms_dict[kt.term] = kt.definition
        
    # Deduplicate Concepts
    for concept in note.core_concepts:
        unique_concepts_dict[concept.concept_name] = concept


# --- EXPORT TO MARKDOWN ---
# Easily change this string for future lectures!
lecture_title = "Computer Networks" 

print(f"Generating Notion-friendly Markdown file for '{lecture_title}'...")

# 1. Build the Markdown String
md_content = f"# 🎓 {lecture_title} Study Guide\n\n"
md_content += f"## 📝 Comprehensive Summary\n{master_summary}\n"

md_content += f"## 🔑 Key Terms\n"
for term, definition in unique_terms_dict.items():
    md_content += f"* **{term}**: {definition}\n"

md_content += f"\n## 📌 Core Concepts\n"
for name, concept in unique_concepts_dict.items():
    md_content += f"### {name}\n"
    md_content += f"{concept.detailed_explanation}\n\n"
    if concept.examples:
        md_content += "**Examples:**\n"
        for ex in concept.examples:
            md_content += f"- {ex}\n"
    md_content += "\n"

# 2. Save it to a file
os.makedirs("../data/outputs", exist_ok=True)
# This dynamically names the file based on your title! (e.g. Computer_Networks_Guide.md)
file_name = lecture_title.replace(" ", "_") + "_Guide.md"
output_path = f"../data/outputs/{file_name}"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(md_content)

print(f"✅ DONE! Your beautiful notes are saved at: {output_path}")

Combining notes and removing duplicates...
Generating Notion-friendly Markdown file for 'Computer Networks'...
✅ DONE! Your beautiful notes are saved at: ../data/outputs/Computer_Networks_Guide.md
